# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [ ]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 50.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=5572c993-19ac-4bb9-97a7-c2c628283f51
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 59.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 233MB/s]


In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)

    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    else:
        delta = torch.zeros_like(x)
    delta = clip_box(x, delta)

    for step in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    x_adv = (x + delta).detach()
    return x_adv



def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)

    if rand_init:
        delta = torch.randn_like(x)
        random_radius = torch.rand(x.shape[0], 1, 1, 1, device=x.device) * eps
        delta = delta / l2_norm(delta).clamp_min(1e-12) * random_radius
    else:
        delta = torch.zeros_like(x)
    delta = clip_box(x, delta)

    for step in range(steps):
        grad = input_grad(model, x, delta, y)
        grad_norm = l2_norm(grad).clamp_min(1e-12)    #avoid divide by zero
        delta = delta + alpha * grad / grad_norm

        delta_norm = l2_norm(delta).clamp_min(1e-12)
        scale_factor = (eps / delta_norm).clamp(max=1.0)
        delta = delta * scale_factor

        delta = clip_box(x, delta)

    x_adv = (x + delta).detach()
    return x_adv


In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (119 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 59.8, 'l1': 22.4, 'union': 22.3}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss

    model.eval() #eval mode
    x_adv = attack(model, x, y, norm)
    model.train() # train mode to getting the training loss
    logits_adv = model(x_adv)
    loss = F.cross_entropy(logits_adv, y)
    return loss

    raise NotImplementedError

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (41 s)
epoch 2: loss 1.181 (84 s)
epoch 3: loss 1.151 (126 s)
epoch 4: loss 1.126 (169 s)
epoch 5: loss 1.097 (211 s)
epoch 6: loss 1.068 (254 s)
epoch 7: loss 1.048 (296 s)
epoch 8: loss 1.030 (338 s)
epoch 9: loss 1.013 (381 s)
epoch 10: loss 1.008 (423 s)
clean 78.4  linf 45.2  l2 57.1  l1 20.2  union 20.1   (129 s)


{'clean': 78.4, 'linf': 45.2, 'l2': 57.1, 'l1': 20.2, 'union': 20.1}

In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.897 (45 s)
epoch 2: loss 0.826 (86 s)
epoch 3: loss 0.787 (130 s)
epoch 4: loss 0.754 (172 s)
epoch 5: loss 0.714 (215 s)
epoch 6: loss 0.675 (257 s)
epoch 7: loss 0.631 (300 s)
epoch 8: loss 0.610 (343 s)
epoch 9: loss 0.591 (385 s)
epoch 10: loss 0.583 (428 s)
clean 82.5  linf 32.3  l2 59.9  l1 28.4  union 24.6   (129 s)


{'clean': 82.5, 'linf': 32.3, 'l2': 59.9, 'l1': 28.4, 'union': 24.6}

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.233 (44 s)
epoch 2: loss 0.881 (86 s)
epoch 3: loss 0.588 (130 s)
epoch 4: loss 0.459 (172 s)
epoch 5: loss 0.346 (215 s)
epoch 6: loss 0.282 (257 s)
epoch 7: loss 0.221 (300 s)
epoch 8: loss 0.195 (343 s)
epoch 9: loss 0.162 (386 s)
epoch 10: loss 0.163 (429 s)
clean 83.7  linf 0.0  l2 0.1  l1 25.9  union 0.0   (131 s)


{'clean': 83.7, 'linf': 0.0, 'l2': 0.1, 'l1': 25.9, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL

    model.eval()
    x_adv_linf = attack(model, x, y, "linf")
    x_adv_l1 = attack(model, x, y, "l1")
    #loss in train mode
    model.train()
    logits_linf = model(x_adv_linf)
    logits_l1 = model(x_adv_l1)

    loss_linf_per_image = F.cross_entropy(logits_linf, y, reduction="none")
    loss_l1_per_image = F.cross_entropy(logits_l1, y, reduction="none")
    max_loss = torch.maximum(loss_linf_per_image,loss_l1_per_image).mean()

    correct_under_linf = (logits_linf.argmax(1) == y).detach()
    if correct_under_linf.sum() == 0:
        kl_loss = torch.zeros((), device=x.device)
    else:
        log_prob_linf = F.log_softmax(logits_linf[correct_under_linf], dim=1)
        log_prob_l1 = F.log_softmax(logits_l1[correct_under_linf], dim=1)
        kl_loss = F.kl_div(log_prob_l1,log_prob_linf,reduction="batchmean",log_target=True)

    total_loss = max_loss + lam * kl_loss
    return total_loss


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor

    g_n_flat = g_n.flatten()
    g_a_flat = g_a.flatten()

    dot_product = (g_n_flat * g_a_flat).sum()
    norm_product = g_n_flat.norm() * g_a_flat.norm()
    cosine_sim = dot_product / (norm_product + 1e-12)

    positive_cosine = cosine_sim.clamp(min=0)
    projected_gradient = positive_cosine * g_n
    return projected_gradient

    raise NotImplementedError

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432 (93 s)
epoch 2: loss 1.330 (189 s)
epoch 3: loss 1.303 (285 s)
epoch 4: loss 1.280 (382 s)
epoch 5: loss 1.254 (478 s)
epoch 6: loss 1.227 (575 s)
epoch 7: loss 1.199 (671 s)
epoch 8: loss 1.186 (768 s)
epoch 9: loss 1.171 (864 s)
epoch 10: loss 1.165 (961 s)
clean 79.3  linf 40.9  l2 61.8  l1 44.6  union 39.1   (137 s)


{'clean': 79.3, 'linf': 40.9, 'l2': 61.8, 'l1': 44.6, 'union': 39.1}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [ ]:
#in each batch, attack with only a random norm (linf/l1)
# The norm that is currently weaker is chosen more often

adaptive_state = {}


def reset_adaptive_state():
    adaptive_state["running_accuracy"] = {"linf": 0.5, "l1": 0.5}
    adaptive_state["chosen_norms"] = []
    adaptive_state["linf_probabilities"] = [] #probability of choosing linf in each batch

reset_adaptive_state()

def adaptive_norm_loss(model, x, y, adaptive=True, momentum=0.9):
    running_accuracy = adaptive_state["running_accuracy"]

    if adaptive:
        error_linf = 1.0 - running_accuracy["linf"]
        error_l1 = 1.0 - running_accuracy["l1"]
        probability_linf = error_linf / (error_linf + error_l1 + 1e-12)
    else:
        probability_linf = 0.5

    if torch.rand(1).item() < probability_linf:
        chosen_norm = "linf"
    else:
        chosen_norm = "l1"

    #attack with the chosen norm
    model.eval()
    x_adv = attack(model, x, y, chosen_norm)
    model.train()
    logits_adv = model(x_adv)
    loss = F.cross_entropy(logits_adv, y)

    batch_accuracy = (logits_adv.argmax(1) == y).float().mean().item()
    old_accuracy = running_accuracy[chosen_norm]
    running_accuracy[chosen_norm] = momentum * old_accuracy + (1 - momentum) * batch_accuracy

    adaptive_state["chosen_norms"].append(chosen_norm)
    adaptive_state["linf_probabilities"].append(probability_linf)
    return loss


def my_loss(model, x, y):
    # TODO: your method
    return adaptive_norm_loss(model, x, y, adaptive=True)


In [ ]:
#model_mine = adv_train(load_base_model(), my_loss)
#test_model(model_mine)

In [ ]:
PART_C_EPOCHS = 10
trained_models = {}
results = {}
time_per_epoch = {}
training_history = {}

def train_and_time(loss_fn, seed, **kwargs):
    start_time = time.time()
    model = adv_train(load_base_model(), loss_fn, epochs=PART_C_EPOCHS, seed=seed, **kwargs)
    seconds_per_epoch = (time.time() - start_time) / PART_C_EPOCHS
    return model, seconds_per_epoch


for seed in [0, 1]:
    run_name = f"mine_seed{seed}"
    print(run_name)
    reset_adaptive_state()  # start every run with a fresh state
    model, seconds_per_epoch = train_and_time(my_loss, seed)

    trained_models[run_name] = model
    time_per_epoch[run_name] = seconds_per_epoch
    training_history[run_name] = {
        "chosen_norms": list(adaptive_state["chosen_norms"]),
        "linf_probabilities": list(adaptive_state["linf_probabilities"]),
    }
    results[run_name] = test_model(model)

mine_seed0
epoch 1: loss 1.321 (40 s)
epoch 2: loss 1.236 (83 s)
epoch 3: loss 1.198 (128 s)
epoch 4: loss 1.146 (173 s)
epoch 5: loss 1.127 (218 s)
epoch 6: loss 1.103 (264 s)
epoch 7: loss 1.078 (309 s)
epoch 8: loss 1.046 (354 s)
epoch 9: loss 1.032 (398 s)
epoch 10: loss 1.024 (444 s)
clean 77.7  linf 41.8  l2 61.9  l1 48.5  union 40.5   (138 s)
mine_seed1
epoch 1: loss 1.326 (45 s)
epoch 2: loss 1.230 (91 s)
epoch 3: loss 1.191 (136 s)
epoch 4: loss 1.151 (181 s)
epoch 5: loss 1.128 (227 s)
epoch 6: loss 1.089 (272 s)
epoch 7: loss 1.074 (317 s)
epoch 8: loss 1.055 (362 s)
epoch 9: loss 1.040 (408 s)
epoch 10: loss 1.043 (453 s)
clean 77.5  linf 42.0  l2 61.7  l1 48.4  union 40.6   (137 s)


In [ ]:
#reuse model_ramp from part b for seed 0
trained_models["ramp_seed0"] = model_ramp
time_per_epoch["ramp_seed0"] = 961 / 10 #prom prev run
results["ramp_seed0"] = test_model(model_ramp)

#train RAMP again with seed=1
print("ramp_seed1")
model, seconds_per_epoch = train_and_time(ramp_loss, 1, gp_beta=0.5)
trained_models["ramp_seed1"] = model
time_per_epoch["ramp_seed1"] = seconds_per_epoch
results["ramp_seed1"] = test_model(model)

clean 79.3  linf 40.9  l2 61.8  l1 44.6  union 39.1   (138 s)
ramp_seed1
epoch 1: loss 1.433 (96 s)
epoch 2: loss 1.333 (192 s)
epoch 3: loss 1.299 (288 s)
epoch 4: loss 1.274 (384 s)
epoch 5: loss 1.248 (480 s)
epoch 6: loss 1.221 (576 s)
epoch 7: loss 1.200 (672 s)
epoch 8: loss 1.186 (768 s)
epoch 9: loss 1.178 (864 s)
epoch 10: loss 1.169 (960 s)
clean 79.3  linf 42.3  l2 62.9  l1 46.6  union 40.7   (137 s)


In [ ]:
print("ablation_fixed50_seed0")
reset_adaptive_state()
model, seconds_per_epoch = train_and_time(adaptive_norm_loss, 0, adaptive=False)

trained_models["ablation_fixed50_seed0"] = model
time_per_epoch["ablation_fixed50_seed0"] = seconds_per_epoch
training_history["ablation_fixed50_seed0"] = {
    "chosen_norms": list(adaptive_state["chosen_norms"]),
    "linf_probabilities": list(adaptive_state["linf_probabilities"]),
}
results["ablation_fixed50_seed0"] = test_model(model)

ablation_fixed50_seed0
epoch 1: loss 1.305 (46 s)
epoch 2: loss 1.218 (91 s)
epoch 3: loss 1.191 (136 s)
epoch 4: loss 1.125 (181 s)
epoch 5: loss 1.127 (226 s)
epoch 6: loss 1.085 (272 s)
epoch 7: loss 1.061 (317 s)
epoch 8: loss 1.044 (362 s)
epoch 9: loss 1.023 (407 s)
epoch 10: loss 1.004 (452 s)
clean 77.3  linf 41.5  l2 62.2  l1 48.8  union 40.2   (137 s)


In [ ]:
results["base"] = test_model(load_base_model())

def mean_over_runs(run_names):
    mean_accuracy = {}
    for metric in ["clean", "linf", "l2", "l1", "union"]:
        values = [results[run_name][metric] for run_name in run_names]
        mean_accuracy[metric] = sum(values) / len(values)
    return mean_accuracy

def mean_time(run_names):
    values = [time_per_epoch[run_name] for run_name in run_names]
    return sum(values) / len(values)

table_rows = [
    ("Base model", results["base"], None),
    ("RAMP (mean of seed 0, 1)", mean_over_runs(["ramp_seed0", "ramp_seed1"]), mean_time(["ramp_seed0", "ramp_seed1"])),
    ("Mine (mean of seed 0, 1)", mean_over_runs(["mine_seed0", "mine_seed1"]), mean_time(["mine_seed0", "mine_seed1"])),
    ("Ablation: fixed 50/50 (seed 0)", results["ablation_fixed50_seed0"], time_per_epoch["ablation_fixed50_seed0"]),
]

print(f"{'model':<32} {'clean':>6} {'linf':>6} {'l2':>6} {'l1':>6} {'union':>6} {'s/epoch':>8}")
for row_name, accuracy, seconds in table_rows:
    if seconds is None:
        seconds_text = "-"
    else:
        seconds_text = f"{seconds:.1f}"
    print(f"{row_name:<32} {accuracy['clean']:>6.1f} {accuracy['linf']:>6.1f} {accuracy['l2']:>6.1f} "
          f"{accuracy['l1']:>6.1f} {accuracy['union']:>6.1f} {seconds_text:>8}")

print()
print("Each single run:")
for run_name in ["ramp_seed0", "ramp_seed1", "mine_seed0", "mine_seed1", "ablation_fixed50_seed0"]:
    accuracy = results[run_name]
    print(f"{run_name:<32} {accuracy['clean']:>6.1f} {accuracy['linf']:>6.1f} {accuracy['l2']:>6.1f} "
          f"{accuracy['l1']:>6.1f} {accuracy['union']:>6.1f} {time_per_epoch[run_name]:>8.1f}")

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (133 s)
model                             clean   linf     l2     l1  union  s/epoch
Base model                         78.7   48.4   59.8   22.4   22.3        -
RAMP (mean of seed 0, 1)           79.3   41.6   62.3   45.6   39.9     96.0
Mine (mean of seed 0, 1)           77.6   41.9   61.8   48.5   40.5     44.8
Ablation: fixed 50/50 (seed 0)     77.3   41.5   62.2   48.8   40.2     45.2

Each single run:
ramp_seed0                         79.3   40.9   61.8   44.6   39.1     96.1
ramp_seed1                         79.3   42.3   62.9   46.6   40.7     96.0
mine_seed0                         77.7   41.8   61.9   48.5   40.5     44.4
mine_seed1                         77.5   42.0   61.7   48.4   40.6     45.3
ablation_fixed50_seed0             77.3   41.5   62.2   48.8   40.2     45.2


In [ ]:
#analysis
batches_per_epoch = len(x_train) // 128  # adv_train uses batch_size=128 and skips the last smaller batch

for run_name in ["mine_seed0", "mine_seed1", "ablation_fixed50_seed0"]:
    chosen_norms = training_history[run_name]["chosen_norms"]
    linf_probabilities = training_history[run_name]["linf_probabilities"]
    print(f"{run_name}:")
    print(f"  {'epoch':>5} {'linf batches (%)':>17} {'mean P(linf)':>13}")

    for epoch in range(PART_C_EPOCHS):
        start_batch = epoch * batches_per_epoch
        end_batch = start_batch + batches_per_epoch
        norms_in_epoch = chosen_norms[start_batch:end_batch]
        probabilities_in_epoch = linf_probabilities[start_batch:end_batch]

        linf_percentage = 100.0 * norms_in_epoch.count("linf") / len(norms_in_epoch)
        mean_probability = sum(probabilities_in_epoch) / len(probabilities_in_epoch)
        print(f"  {epoch + 1:>5} {linf_percentage:>17.1f} {mean_probability:>13.2f}")
    print()

mine_seed0:
  epoch  linf batches (%)  mean P(linf)
      1              51.3          0.49
      2              60.3          0.51
      3              62.8          0.51
      4              44.9          0.52
      5              55.1          0.54
      6              57.7          0.54
      7              59.0          0.54
      8              52.6          0.54
      9              44.9          0.54
     10              51.3          0.55

mine_seed1:
  epoch  linf batches (%)  mean P(linf)
      1              47.4          0.49
      2              50.0          0.52
      3              60.3          0.53
      4              55.1          0.53
      5              56.4          0.54
      6              51.3          0.53
      7              53.8          0.53
      8              57.7          0.54
      9              56.4          0.54
     10              66.7          0.54

ablation_fixed50_seed0:
  epoch  linf batches (%)  mean P(linf)
      1              37.2     

In [ ]:
if results["mine_seed0"]["union"] >= results["mine_seed1"]["union"]:
    best_run_name = "mine_seed0"
else:
    best_run_name = "mine_seed1"
print("best model:", best_run_name)

print("default evaluation:")
print(results[best_run_name])

print("strong evaluation (steps=200, restarts=5):")
results[best_run_name + "_strong"] = test_model(trained_models[best_run_name], steps=200, restarts=5)

best model: mine_seed1
default evaluation:
{'clean': 77.5, 'linf': 42.0, 'l2': 61.7, 'l1': 48.4, 'union': 40.6}
strong evaluation (steps=200, restarts=5):
clean 77.5  linf 41.6  l2 61.6  l1 46.5  union 39.5   (1363 s)
